In [1]:
import os
import glob
import cv2
import numpy as np
import kagglehub

In [2]:
#Fetching kaggle path
XRAY_DATASET_PATH = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
MRI_DATASET_PATH = kagglehub.dataset_download("navoneel/brain-mri-images-for-brain-tumor-detection")

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
Using Colab cache for faster access to the 'brain-mri-images-for-brain-tumor-detection' dataset.


In [3]:
print(f"-> X-Ray path resolved to: {XRAY_DATASET_PATH}")
print(f"-> MRI path resolved to: {MRI_DATASET_PATH}")

-> X-Ray path resolved to: /kaggle/input/chest-xray-pneumonia
-> MRI path resolved to: /kaggle/input/brain-mri-images-for-brain-tumor-detection


In [4]:
OUTPUT_DIR = "./transformed_repository"
TARGET_SIZE = (512, 512)
INTERPOLATION_METHOD = cv2.INTER_CUBIC

In [5]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [6]:
INTERPOLATION_METHODS = {
    "nearest": cv2.INTER_NEAREST,
    "linear": cv2.INTER_LINEAR,
    "cubic": cv2.INTER_CUBIC,
    "area": cv2.INTER_AREA,
    "lanczos4": cv2.INTER_LANCZOS4
}

In [7]:
XRAY_OUTPUT = os.path.join(OUTPUT_DIR, "xray_processed")
MRI_OUTPUT = os.path.join(OUTPUT_DIR, "mri_processed")

os.makedirs(XRAY_OUTPUT, exist_ok=True)
os.makedirs(MRI_OUTPUT, exist_ok=True)

SELECTED_INTERPOLATION = "cubic" # Using 'cubic' as defined by INTERPOLATION_METHOD initially

In [8]:
#loading images directly from the kaggle

def load_sample_images():
    raw_images = []


     # 15 images from Chest X-Ray in .jpeg format
    xray_pattern = os.path.join(XRAY_DATASET_PATH, "**", "*.jpeg")
    xray_files = glob.glob(xray_pattern, recursive=True)[:15]
    for f in xray_files:
        raw_images.append(("Xray", f))
    # 15 form Mri in .jpeg format
    mri_pattern = os.path.join(MRI_DATASET_PATH, "**", "*.jpg")
    mri_files = glob.glob(mri_pattern, recursive=True)[:15]
    for f in mri_files:
        raw_images.append(("MRI", f))

In [15]:
# Starting main part normalizer

def normalize_scan(image_path, target_size=(512, 512), interpolation=cv2.INTER_CUBIC):


# fetching image from dataset

    img = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
    if img is None:
        return None, "Failed to read image"

# converting into greyscale


    if len(img.shape) == 3:
        img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        img_gray = img

#Aspect Ratio Preserving Resize

    h, w = img_gray.shape[:2]

    target_w, target_h = TARGET_SIZE

    scale = min(target_w / w, target_h / h)

    new_w = int(w * scale)
    new_h = int(h * scale)

    resized = cv2.resize(
        img_gray,
        (new_w, new_h),
        interpolation=interpolation
    )
# Padding for maintaing exact size

    pad_x = target_w - new_w
    pad_y = target_h - new_h

    top = pad_y // 2
    bottom = pad_y - top

    left = pad_x // 2
    right = pad_x - left

    padded = cv2.copyMakeBorder(
        resized,
        top,
        bottom,
        left,
        right,
        cv2.BORDER_CONSTANT,
        value=0
    )

# Reducting the noise

    denoised = cv2.GaussianBlur(
        padded,
        (5, 5),
        0
    )
# CLAHE Enhancement

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )
    enhanced = clahe.apply(denoised)

 # Min-Max Normalization

    img_float = enhanced.astype(np.float32)
    img_min = np.min(img_float)
    img_max = np.max(img_float)

    if img_max - img_min != 0:
        normalized_img = (img_float - img_min) / (img_max - img_min)
    else:
        normalized_img = img_float

    return normalized_img, "Success"

In [16]:
# processing dataset fun

def process_dataset(
    input_folder,
    output_folder,
    max_images=100,
    interpolation_name="linear"
):

    interpolation = INTERPOLATION_METHODS[
        interpolation_name
    ]

    image_count = 0

    print(f"\nProcessing images from: {input_folder}")

    for root, dirs, files in os.walk(input_folder):

        for file in files:

            if file.lower().endswith(
                (".jpg", ".jpeg", ".png")
            ):

                if image_count >= max_images:
                    return

                image_path = os.path.join(root, file)

                save_name = f"{image_count+1}_{file}"

                save_path = os.path.join(
                    output_folder,
                    save_name
                )

                processed_image, status_message = normalize_scan(
                    image_path,
                    TARGET_SIZE,
                    interpolation
                )

                if processed_image is not None:
                    # Save the processed image. `cv2.imwrite` expects a 0-255 uint8 image,
                    # so we need to convert the float32 [0,1] image back.
                    cv2.imwrite(save_path, (processed_image * 255).astype(np.uint8))
                    image_count += 1
                else:
                    print(f"Error processing {image_path}: {status_message}")

    print(f"Total Processed Images: {image_count}")

In [17]:
# chest xray image folder

xray_images_folder = os.path.join(
    XRAY_DATASET_PATH,
    "chest_xray"
)

mri_images_folder = MRI_DATASET_PATH

In [18]:
# process the xray image
process_dataset(
    input_folder=xray_images_folder,
    output_folder=XRAY_OUTPUT,
    max_images=15,
    interpolation_name=SELECTED_INTERPOLATION
)


Processing images from: /kaggle/input/chest-xray-pneumonia/chest_xray


In [19]:
# process the mri
process_dataset(
    input_folder=mri_images_folder,
    output_folder=MRI_OUTPUT,
    max_images=15,
    interpolation_name=SELECTED_INTERPOLATION
)


Processing images from: /kaggle/input/brain-mri-images-for-brain-tumor-detection
